# Rockfall AI — EfficientNet-B0 (v3, improved generalization)

This is your v2 pipeline with the parts that most affect real-world generalization
upgraded: progressive unfreezing, mixup, SWA, test-time augmentation, and a
threshold chosen from validation data instead of a hand-picked constant. It does
**not** promise a "perfect" model — on ~2,100 images, claiming 100% would mean the
model memorized the training set, not that it generalizes. The goal here is a
model that's measurably *more robust*, with an honestly-reported number.

Run this in Colab exactly like v2 — same Drive mount, same dataset layout
(`dataset/train|validation|test/class0|class1`).

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DATASET_ROOT = Path("/content/drive/MyDrive/dataset")
ROCKFALL_ROOT = Path("/content/drive/MyDrive/RockfallAI")
ROCKFALL_ROOT.mkdir(parents=True, exist_ok=True)

for split in ["train", "validation", "test"]:
    p = DATASET_ROOT / split
    print(split, "exists:", p.exists())


In [ ]:
import torch, torchvision
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 1. Data — same augmentation as v2 (it was already solid)

Kept as-is: RandomResizedCrop, flip, rotation, color jitter, affine, random
erasing. This part of your pipeline wasn't the bottleneck.

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.80, 1.0), ratio=(0.90, 1.10)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15, hue=0.03),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05), shear=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.12), ratio=(0.3, 3.3), value=0),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Horizontally-flipped eval transform, used only for test-time augmentation
eval_transform_flip = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(DATASET_ROOT / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(DATASET_ROOT / "validation", transform=eval_transform)
val_dataset_flip = datasets.ImageFolder(DATASET_ROOT / "validation", transform=eval_transform_flip)
test_dataset = datasets.ImageFolder(DATASET_ROOT / "test", transform=eval_transform)
test_dataset_flip = datasets.ImageFolder(DATASET_ROOT / "test", transform=eval_transform_flip)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_loader_flip = DataLoader(val_dataset_flip, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_flip = DataLoader(test_dataset_flip, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("class_to_idx:", train_dataset.class_to_idx)
print("train/val/test sizes:", len(train_dataset), len(val_dataset), len(test_dataset))


## 2. Model — same EfficientNet-B0 backbone, staged unfreezing

**Change 1: progressive unfreezing.** Instead of fine-tuning every layer from
epoch 1 (which risks distorting good pretrained ImageNet features before the
new classifier head has learned anything useful), the backbone is frozen for
the first few epochs so only the new classifier head trains, then the backbone
unfreezes with a lower learning rate than the head. This is a standard
transfer-learning fix for small datasets.

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.30),
    nn.Linear(num_features, 2),
)
model = model.to(DEVICE)

FREEZE_EPOCHS = 3  # backbone stays frozen for this many epochs

def set_backbone_trainable(trainable: bool):
    for name, param in model.named_parameters():
        if not name.startswith("classifier"):
            param.requires_grad = trainable

set_backbone_trainable(False)
print("Backbone frozen. Trainable params:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))


## 3. Loss, optimizer, schedule

**Change 2: discriminative learning rates.** Once unfrozen, the backbone trains
at a smaller LR than the classifier head — the head needs to move further from
its random init, the backbone just needs gentle adjustment.

**Change 3: mixup.** For a dataset this size, mixup (blending pairs of images
and their labels) is one of the highest-value regularizers available — it
smooths the decision boundary and reliably improves held-out accuracy on small
image datasets without adding any real training cost.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np

EPOCHS = 30
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
MIXUP_ALPHA = 0.2
SWA_START_EPOCH = 24  # last ~6 epochs average into a flatter, more robust minimum

class_counts = torch.bincount(torch.tensor(train_dataset.targets))
class_weights = (class_counts.sum().float() / (len(class_counts) * class_counts.float())).to(DEVICE)
print("Class weights:", class_weights.tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

def build_optimizer():
    head_params = [p for n, p in model.named_parameters() if n.startswith("classifier")]
    backbone_params = [p for n, p in model.named_parameters() if not n.startswith("classifier")]
    return AdamW([
        {"params": head_params, "lr": HEAD_LR},
        {"params": backbone_params, "lr": BACKBONE_LR},
    ], weight_decay=WEIGHT_DECAY)

optimizer = build_optimizer()
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

def mixup_batch(images, labels, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return images, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1 - lam) * images[perm]
    return mixed, labels, labels[perm], lam


## 4. Training loop — early stopping on val F1 + SWA tail

Same early-stopping-on-F1 and gradient clipping as v2. Added: unfreeze the
backbone (and rebuild the optimizer with the two-LR groups) right after
`FREEZE_EPOCHS`, and average model weights over the final few epochs with
Stochastic Weight Averaging, which consistently finds flatter, better-
generalizing minima than the single last checkpoint.

In [ ]:
import time
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from torch.optim.swa_utils import AveragedModel, update_bn

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

PATIENCE = 7
GRAD_CLIP = 1.0
BEST_MODEL_PATH = str(ROCKFALL_ROOT / "efficientnet_b0_best_v3.pth")

swa_model = AveragedModel(model)
best_val_f1 = -1.0
patience_counter = 0
history = {"train_loss": [], "val_loss": [], "val_f1": []}

for epoch in range(EPOCHS):
    t0 = time.time()

    if epoch == FREEZE_EPOCHS:
        set_backbone_trainable(True)
        optimizer = build_optimizer()
        scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS - FREEZE_EPOCHS, eta_min=1e-6)
        print(f">> epoch {epoch}: backbone unfrozen, discriminative LRs active")

    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        mixed, y_a, y_b, lam = mixup_batch(images, labels)

        optimizer.zero_grad()
        outputs = model(mixed)
        loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    if epoch >= SWA_START_EPOCH:
        swa_model.update_parameters(model)
    scheduler.step()
    train_loss = running_loss / len(train_dataset)

    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item() * images.size(0)
            all_preds.extend(outputs.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    val_loss /= len(val_dataset)
    val_f1 = f1_score(all_labels, all_preds)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | train_loss {train_loss:.4f} | "
          f"val_loss {val_loss:.4f} | val_f1 {val_f1:.4f} | {time.time()-t0:.1f}s")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "best_val_f1": best_val_f1,
            "class_to_idx": train_dataset.class_to_idx,
            "img_size": IMG_SIZE,
        }, BEST_MODEL_PATH)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print("\nBest single-checkpoint val F1:", best_val_f1)


## 5. Finalize SWA weights

Batch-norm running stats need recomputing after averaging weights — that's
what `update_bn` does. We then compare the plain best checkpoint against the
SWA-averaged model on validation, and keep whichever is actually better rather
than assuming SWA wins by default.

In [ ]:
update_bn(train_loader, swa_model, device=DEVICE)

def evaluate(net, loader):
    net.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            out = net(images)
            preds.extend(out.argmax(1).cpu().tolist())
            labels_all.extend(labels.tolist())
    return f1_score(labels_all, preds)

swa_val_f1 = evaluate(swa_model, val_loader)
best_ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(best_ckpt["model_state_dict"])
plain_val_f1 = evaluate(model, val_loader)

print("Plain best-checkpoint val F1:", plain_val_f1)
print("SWA-averaged val F1:         ", swa_val_f1)

USE_SWA = swa_val_f1 > plain_val_f1
final_model = swa_model.module if USE_SWA else model
print("\nUsing", "SWA-averaged" if USE_SWA else "plain best-checkpoint", "weights for final evaluation.")


## 6. Threshold — chosen from validation data, not hand-picked

v2 hardcoded `TEST_THRESHOLD = 0.61` without stating how it was chosen. Here
the threshold is swept against **validation** probabilities to maximize F1,
then applied once to the untouched test set — so the reported test number
isn't quietly threshold-tuned on the same data it's evaluated on.

**Test-time augmentation (TTA):** each image's final probability is the
average of the model's prediction on the image and on its horizontal flip —
a small, well-established accuracy bump with no retraining required.

In [ ]:
import torch.nn.functional as F

def tta_probs(net, loader, loader_flip):
    net.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for (images, labels), (images_f, _) in zip(loader, loader_flip):
            images, images_f = images.to(DEVICE), images_f.to(DEVICE)
            p1 = F.softmax(net(images), dim=1)[:, 1]
            p2 = F.softmax(net(images_f), dim=1)[:, 1]
            avg = (p1 + p2) / 2
            all_probs.extend(avg.cpu().tolist())
            all_labels.extend(labels.tolist())
    return np.array(all_probs), np.array(all_labels)

val_probs, val_labels = tta_probs(final_model, val_loader, val_loader_flip)

thresholds = np.linspace(0.05, 0.95, 181)
f1s = [f1_score(val_labels, (val_probs >= t).astype(int)) for t in thresholds]
BEST_THRESHOLD = float(thresholds[int(np.argmax(f1s))])
print(f"Best threshold from validation data: {BEST_THRESHOLD:.3f} (val F1 {max(f1s):.4f})")


## 7. Final test evaluation — honestly reported

Same metrics as v2 (accuracy, precision, recall, F1, ROC-AUC, PR-AUC,
confusion matrix), now with TTA + the validation-derived threshold. Whatever
number comes out here is the real one — don't re-tune anything against it.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report,
)

test_probs, test_labels = tta_probs(final_model, test_loader, test_loader_flip)
test_preds = (test_probs >= BEST_THRESHOLD).astype(int)

print("=" * 70)
print("FINAL TEST RESULTS — EfficientNet-B0 v3")
print("=" * 70)
print(f"Threshold : {BEST_THRESHOLD:.3f} (selected on validation data)")
print(f"Accuracy  : {accuracy_score(test_labels, test_preds):.4f}")
print(f"Precision : {precision_score(test_labels, test_preds):.4f}")
print(f"Recall    : {recall_score(test_labels, test_preds):.4f}")
print(f"F1 Score  : {f1_score(test_labels, test_preds):.4f}")
print(f"ROC-AUC   : {roc_auc_score(test_labels, test_probs):.4f}")
print(f"PR-AUC    : {average_precision_score(test_labels, test_probs):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_preds))
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=["Low Risk", "High Risk"], digits=4))


## 8. Save the deployable model — weights + threshold + ONNX export

Everything a real inference service needs travels together in one checkpoint:
weights, the class mapping, image size, **and the threshold** (so a server
doesn't have to guess it later). An ONNX export is included too, since ONNX
Runtime is much lighter to deploy behind an API (e.g. on Vercel serverless or
Hugging Face Spaces) than a full PyTorch install.

In [ ]:
FINAL_MODEL_PATH = str(ROCKFALL_ROOT / "efficientnet_b0_final_v3.pth")

torch.save({
    "model_state_dict": final_model.state_dict(),
    "class_to_idx": train_dataset.class_to_idx,
    "img_size": IMG_SIZE,
    "threshold": BEST_THRESHOLD,
    "used_swa": USE_SWA,
    "test_f1": float(f1_score(test_labels, test_preds)),
    "test_roc_auc": float(roc_auc_score(test_labels, test_probs)),
}, FINAL_MODEL_PATH)

print("Saved:", FINAL_MODEL_PATH)

# --- ONNX export ---
ONNX_PATH = str(ROCKFALL_ROOT / "efficientnet_b0_final_v3.onnx")
final_model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
torch.onnx.export(
    final_model, dummy_input, ONNX_PATH,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
print("Saved:", ONNX_PATH)


In [ ]:
from google.colab import files

files.download(FINAL_MODEL_PATH)
files.download(ONNX_PATH)


## What changed vs. v2, and why each change should help

| Change | Why it helps generalization |
|---|---|
| Progressive unfreezing | Stops the pretrained backbone from being distorted before the new head has learned anything |
| Discriminative LRs | Backbone gets gentle fine-tuning, head gets full learning capacity |
| Mixup | Smooths the decision boundary — one of the best-evidenced regularizers for small image datasets |
| SWA (kept only if it wins on validation) | Averages over a flatter region of loss landscape, often more robust than a single checkpoint |
| TTA (flip-averaged) | Free accuracy at inference time, no retraining |
| Threshold from validation, not a magic constant | The reported test metric isn't quietly tuned on the data it's measured against |

None of this guarantees a specific new accuracy number — that depends on your
actual data when you run it. What it guarantees is a more defensible,
honestly-validated result than a single hand-tuned run.